# ProteomicsCoPYlot notebook example

A complete workflow on the bundled HEK293 example (WT vs GRK, five replicates each),
with the same functions the app uses: completeness, normalization, filtering,
imputation, QC, differential testing (Student, Welch, paired), ANOVA, GO
over-representation, PCA scores, the phosphosite collapse and single-protein views.

Install the package first (`python -m pip install ".[bio]"` from a checkout).

In [ ]:
try:
    from proteomicscopylot.notebook_functions import *
except ModuleNotFoundError:  # a checkout that is not installed: use its folder
    import sys
    from pathlib import Path
    sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "proteomicscopylot").is_dir())))
    from proteomicscopylot.notebook_functions import *

## Load the example

The protein and phosphosite tables are Spectronaut reports on the log2 scale.

In [ ]:
data = load_example_data("protein")
meta = load_example_data("protein_metadata")
sites = load_example_data("phospho")
meta_sites = load_example_data("phospho_metadata")
conditions = ["WT", "GRK"]
print(data.shape, sites.shape)
meta.head()

# Protein

## Completeness

In [ ]:
fig = missing_value_plot(data, meta, data_scale="log2")

In [ ]:
fig = missing_value_heatmap(data, meta, id=False, data_scale="log2")

In [ ]:
missing_sample_table(data, meta, id=False, data_scale="log2")

## Filtering and normalization

As in the app, the valid-value filter decides the rows first, then the
normalization is fitted on those rows and applied to every row, so rows with many
missing values (mostly low-abundance, missing unevenly between groups) never set
the reference. Here: at least 3 valid values in each condition, then median
centring. The app's default is no filter; choose your own.

In [ ]:
kept = filter_data(data, meta, 3, mode="per_group")
normalized = normalize_data(data, meta, "median", fit_rows=kept.index)
filtered = normalized.loc[kept.index]
print(len(data), "proteins,", len(filtered), "after filtering")

## Imputation

The default draws missing values around the 1% quantile of each sample's observed
values with 0.3 times their standard deviation; `method="numerical"` uses the Perseus
downshift instead. Keep the non-imputed matrix: the tests below count how many
imputed values each result rests on when you pass it as `observed`.

In [ ]:
imputed = impute_values(filtered, meta, seed=1337)
fig = impute_values(filtered, meta, ret=3)

## QC

QC describes the normalized, unfiltered matrix. CV and abundance are defined on
linear intensities, so they get the back-transformed values; correlation uses log2.

In [ ]:
linear = inverse_log2_transform_data(normalized, meta)
fig = coverage_plot(normalized, meta, id=False, data_scale="log2")

In [ ]:
fig = boxplot_int(normalized, meta)

In [ ]:
fig = histo_int(normalized, meta)

In [ ]:
fig = cov_plot(linear, meta, data_scale="linear")

In [ ]:
fig = abundance_plot(linear, meta, workflow="Protein", data_scale="linear")

In [ ]:
fig = corr_plot(normalized, meta, id=False, data_scale="log2")

In [ ]:
fig = pca_plot(imputed, meta)
pca_scores(imputed, meta).head()

## Differential abundance

log2 fold changes are condition 1 minus condition 2, so positive values are higher in WT.
`test_type` chooses Student's pooled-variance test (`"unpaired"`), Welch's test
(`"unpaired_welch"`) or the paired test (`"paired"`). P-values are
Benjamini-Hochberg adjusted unless `uncorrected=True`.

In [ ]:
volcano = volcano_data_f(filtered, meta, "WT", "GRK", workflow="Protein", test_type="unpaired")
volcano["significance"].value_counts()

In [ ]:
fig = volcano_plot(volcano, "WT", "GRK")
fig.show()

In [ ]:
welch = volcano_data_f(filtered, meta, "WT", "GRK", workflow="Protein", test_type="unpaired_welch")
welch["significance"].value_counts()

In [ ]:
# With imputed values, pass the non-imputed matrix as `observed` to count them per row.
volcano_imputed = volcano_data_f(imputed, meta, "WT", "GRK", workflow="Protein", observed=filtered)
volcano_imputed[["ProteinNames", "log2FC", "adj_pval", "imputed_values"]].head()

A paired test needs a `pair_id` column in the metadata that links each sample to its
partner in the other condition (for example the same donor or culture). This example
is not paired, so the call is only shown:

```python
paired_meta = meta.assign(pair_id=["1", "2", "3", "4", "5"] * 2)  # your real pairing
paired = volcano_data_f(filtered, paired_meta, "WT", "GRK", test_type="paired")
```

## ANOVA

One-way ANOVA across all conditions in the metadata; with two conditions it matches
the Student t-test.

In [ ]:
anova = anova_data_f(filtered, meta)
anova.head()

In [ ]:
fig = anova_plot(anova, meta)

## GO over-representation

Genes of the significant proteins against the tested proteins as background.

In [ ]:
genes = different_genes(filtered, meta, "WT", "GRK", workflow="Protein")
ora = enrichment_data(genes["Upregulated"] + genes["Downregulated"], background=genes["Background"])
ora.head()

In [ ]:
fig = enrichment_analysis(genes["Upregulated"] + genes["Downregulated"], background=genes["Background"])

## Single protein and comparison

In [ ]:
protein_id = str(filtered["ProteinNames"].dropna().iloc[0])
fig = boxplot_int_single_prot(filtered.set_index("ProteinNames", drop=False), meta, protein_id, dots=True)

In [ ]:
fig = compare_prot_line(filtered, meta, conditions, [protein_id], workflow="Protein", id=False)

In [ ]:
fig = correlation_xy_intensity(normalized, meta, "WT", "GRK")

In [ ]:
fig = overlap_plot_condition(normalized, meta, "WT", "GRK", data_scale="log2")

# Phosphosites

In [ ]:
fig = simple_phos_site_plot(sites)

In [ ]:
fig = phossite_coverage_plot(sites, meta_sites, id=False)

In [ ]:
fig = sty_plot(sites)

In [ ]:
fig = missing_value_plot(sites, meta_sites, data_scale="log2")

In [ ]:
sites_filtered = filter_data(sites, meta_sites, 3, mode="per_group")
site_volcano = volcano_data_f(sites_filtered, meta_sites, "WT", "GRK", workflow="Phosphosite")
fig = volcano_plot(site_volcano, "WT", "GRK")
fig.show()

In [ ]:
site_id = str(sites_filtered["PTM_Collapse_key"].dropna().iloc[0])
fig = boxplot_int_single_prot(sites_filtered.set_index("PTM_Collapse_key", drop=False), meta_sites, site_id, dots=True)

In [ ]:
protein_of_site = str(sites_filtered["Protein_group"].dropna().iloc[0])
fig = phossite_on_prot_regulation(sites_filtered, meta_sites, protein_of_site, conditions=conditions, id=False)

`filter_localization(sites, 0.75)` keeps only Class I sites (localization probability
of at least 0.75); `transform_phosprot` takes the same cutoff as `localization_cutoff`.

# Phosphoproteins

Sites are collapsed per protein. The default `robust_site_median` centres every site
on log2 and takes the protein median; the result keeps per-sample site-coverage
columns.

In [ ]:
phosprot = transform_phosprot(sites, meta_sites, mode="robust_site_median", data_scale="log2")
phosprot_filtered = filter_data(phosprot, meta_sites, 3, mode="per_group")
print(len(phosprot), "phosphoproteins,", len(phosprot_filtered), "after filtering")

In [ ]:
fig = phossite_dist(phosprot, cutoff=0.01)

In [ ]:
phosprot_volcano = volcano_data_f(phosprot_filtered, meta_sites, "WT", "GRK", workflow="Phosphoprotein")
fig = volcano_plot(phosprot_volcano, "WT", "GRK")
fig.show()

In [ ]:
phosprot_id = str(phosprot_filtered["Phosphoprotein"].dropna().iloc[0])
fig = compare_prot_line(phosprot_filtered, meta_sites, conditions, [phosprot_id], workflow="Phosphoprotein", id=False)